# Revision Workshop 1 — The Anatomy of an Estimator

### Objectives
- Understand what an sklearn “estimator” is
- Learn the structure of the `fit → predict → evaluate` workflow
- Get comfortable exploring model attributes and methods
- See what happens “under the hood”


## Part 0 — Warm-up: recall from lectures

1. What does “training a model” mean in machine learning?

2. What’s the difference between “training data” and “test data”?

3. Which part of the code usually “learns” the model parameters?

_(discuss briefly before coding.)_

**Answers (Part 0):**
1. **Training a model** means using data to estimate the parameters of a model so that it can map inputs (X) to outputs/targets (y), usually by minimising a loss.
2. **Training data** is used to learn/fit the model; **test data** is held out and used only to evaluate how well the trained model generalises to unseen data.
3. In sklearn, the part of the code that actually *learns* the parameters is the call to `model.fit(X_train, y_train)`.

# Part 1 - The simplest possible regression

In [5]:
from sklearn.linear_model import LinearRegression
import numpy as np

X = np.array([[1], [2], [3], [4]])
y = np.array([2, 4, 6, 8])

model = LinearRegression()
model.fit(X, y)
print("Predictions for [5, 6]:", model.predict([[5], [6]]))

Predictions for [5, 6]: [10. 12.]


## Questions

1. What does `.fit()` do here?

2. What is returned by `.predict()`?

3. What would happen if you called `.predict()` before `.fit()`? (if you don't know the answer, try it!)

**Answers — Questions:**
1. `.fit()` makes the estimator look at the training data (`X`, `y`) and **learn** the parameters (for linear regression: slope(s) and intercept).
2. `.predict()` returns a NumPy array with the **predicted outputs** for the inputs you pass in.
3. If you call `.predict()` before `.fit()`, sklearn will raise a `NotFittedError` (because the model has no learned parameters yet).

## Part 2 — Inspecting what was learned

In [6]:
print("Coefficient:", model.coef_)
print("Intercept:", model.intercept_)

Coefficient: [2.]
Intercept: -1.7763568394002505e-15


**Exercises**

1. Compute the predicted value for `x=10` manually using `y = coef*x + intercept`.

2. Verify that it matches `.predict([[10]])`.

3. What do `coef_` and `intercept_` represent geometrically?

**Answers — Part 2 Exercises:**
1. The manual prediction is `y = model.coef_[0] * 10 + model.intercept_`.
2. This should match `model.predict([[10]])` up to floating point precision, because they are the same formula.
3. `coef_` is the **slope** (or slopes, one per feature) of the learned linear function; `intercept_` is the **bias/offset** — the value of `y` when all features are 0.

# Part 3 — Slightly more complex: multiple features

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split

X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)

model = LinearRegression()

model.fit(X_train, y_train)


LinearRegression()

**Exercises**

1. How many coefficients are there now? `(len(model.coef_))`

2. What’s the shape of `X_train`?

3. Compute predictions and evaluate performance using R² and MSE.

**Answers — Multiple Features:**
1. For the diabetes dataset, `len(model.coef_)` is **10**, because the dataset has 10 features.
2. With `train_test_split(..., random_state=0)`, `X_train.shape` is **(331, 10)** and `X_test.shape` is **(111, 10)**.
3. Example evaluation (with `LinearRegression`): R² ≈ **0.359** and MSE ≈ **3180.16** for the test set (your exact numbers may differ slightly due to floating point).

In [8]:
from sklearn.metrics import r2_score, mean_squared_error

y_pred = model.predict(X_test)
print("R²:", r2_score(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))

R²: 0.35940880381777063
MSE: 3180.1596481558454


# Part 4 — Comparing estimators

Let's try `Ridge()` and `Lasso()` on the same data.

In [9]:
from sklearn.linear_model import Ridge, Lasso

X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)

ridge = Ridge(alpha=1.0).fit(X_train, y_train)
ridge_r2 = ridge.score(X_test, y_test)

lasso = Lasso(alpha=0.001, max_iter=10000).fit(X_train, y_train)
lasso_r2 = lasso.score(X_test, y_test)

Find the value of the coefficients for each model and fill in this table:

| Model | R² | # Non-zero Coefficients |
|--------|----|-----------------------|
| Linear regression with ridge regularisation | ... | ... |
| Linear regression with lasso regularisation | ... | ... |

In [12]:
print(ridge.coef_)
print(ridge_r2)
print(lasso.coef_)
print(lasso_r2)

[  21.2000037   -60.47643128  302.87680545  179.41025522    8.90955986
  -28.80767284 -149.30718882  112.67212884  250.53509464   99.57769388]
0.3569596077458861
[ -42.20729813 -207.49152165  594.23158398  301.66170923 -515.42397264
  226.1309937   -28.5107603   129.42813254  686.60923964   28.08147747]
0.3587582202150228


**Possible filled table (with the provided code):**

| Model              | R² (test) | # Non-zero Coefficients |
|--------------------|-----------|--------------------------|
| LinearRegression   | ~0.3594   | 10                       |
| Ridge(alpha=1.0)   | ~0.3570   | 10                       |
| Lasso(alpha=0.001) | ~0.3588   | 10                       |

These are close in performance because the dataset is small/moderate and regularisation is not very strong.


*Discussion:*
- What changes when regularisation is applied?
- Why might one prefer Ridge or Lasso?


**Discussion — sample answer:**
- Regularisation (Ridge/Lasso) usually **shrinks** coefficients towards zero, which can slightly reduce variance and improve stability, especially with correlated features.
- **Ridge** is good when we believe all features matter but we want to control magnitude; **Lasso** is useful when we want some feature selection (it can drive some coefficients to 0 if alpha is larger).

# Part 5 - Translating and explaining

Without running the code, explain what every cell is doing in English.

**How to approach this part:**
You should read each code cell that follows and restate it in English. Below are model answers for each of them.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

What is the cell above doing? What does `test_size` mean? What does `random_state` control?

**Answer:**
The cell splits the full dataset `(X, y)` into two parts: a training set and a test set.
- `test_size=0.25` means **25%** of the data goes to the test set and 75% to the training set.
- `random_state=42` fixes the randomness so we get the **same split every time** we run the code.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

What is the cell above doing? Why do we call `fit_transform()` on the training data and only `transform()` on the test data?

**Answer:**
This cell is **scaling/standardising** the features.
- `scaler.fit_transform(X_train)` computes the mean and std **on the training data only** and applies the scaling to the training data.
- `scaler.transform(X_test)` applies **the same scaling** (same mean and std) to the test data.
- We do **not** fit on the test data because that would leak information from the test set into the training process.

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

scores = cross_val_score(LogisticRegression(), X, y, cv=5)

print(scores.mean())

What is the cell above doing? What happens inside `cross_val_score`? What does `cv=5` mean?

**Answer:**
This cell is doing **k-fold cross-validation** with `k=5`.
- `cross_val_score(LogisticRegression(), X, y, cv=5)` splits the data into 5 folds;
- for each fold, it trains a fresh `LogisticRegression` on 4 folds and tests on the remaining 1 fold;
- it returns 5 scores (one per fold).
- `cv=5` literally means “use 5 folds”.

In [ ]:
from sklearn.pipeline import make_pipeline

pipe = make_pipeline(StandardScaler(), LogisticRegression())

pipe.fit(X_train, y_train)

What do you think the code above is doing?

_Answer:_ The code is creating a pipeline that will be run all at once. The pipeline is made of a standard scaler (standardising the data) and a logistic regression classifier.

__________________

### Now let's translate the other way round.

Write sklearn code that matches these plain-English instructions:

1. Split `X` and `y` into 70% training and 30% test sets using seed `123`.

2. Train a logistic regression classifier and compute its accuracy on the test set.

3. Scale the data using a standard scaler before training.
 
4. Perform 10-fold cross-validation and print the mean accuracy.

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# 1. load or assume X, y already exist
X, y = load_diabetes(return_X_y=True)

# 1. split 70/30 with seed 123
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=123
)

# 2 & 3. scale and train
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

clf = LogisticRegression(max_iter=200)
clf.fit(X_train_scaled, y_train)

from sklearn.metrics import accuracy_score
y_pred = clf.predict(X_test_scaled)
acc = accuracy_score(y_test, y_pred)
print('Test accuracy:', acc)

# 4. 10-fold CV on the scaled training data
scores = cross_val_score(
    LogisticRegression(max_iter=200),
    X_train_scaled,
    y_train,
    cv=10
)
print('Mean CV accuracy:', scores.mean())


Now repeat, but using a Pipeline instead of separate steps.

In [ ]:
# Sample pipeline solution:

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123)

# build pipeline: scale → logistic regression
pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=200))
pipe.fit(X_train, y_train)
print('Pipeline test accuracy:', pipe.score(X_test, y_test))

# 10-fold CV directly on the pipeline
cv = KFold(n_splits=10)
scores = cross_val_score(pipe, X_train, y_train, cv = cv)
print('Pipeline CV mean accuracy:', scores.mean())


Pipeline test accuracy: 0.0
Pipeline CV mean accuracy: 0.00967741935483871
